# Fine-tuning experimental do NCM para Python

Este notebook adapta o **Next-Concept Modeling (NCM)** oficial ao laboratorio existente.

Estratégia em duas etapas:
- Stage 1: adapta o VQ-VAE XSum a um corpus maior de Python, preservando whitespace e estrutura;
- Stage 2: faz fine-tuning do gerador sobre o novo VQ-VAE;
- geração iterativa em blocos nativos de 48 tokens, chegando a 144 tokens;
- amostragem de varios candidatos seguida de `ast.parse`.

Por padrao, `RUN_VQVAE_TRAINING=False` e `RUN_GENERATOR_TRAINING=False`. Execute o Stage 1, valide a reconstrução e depois execute o Stage 2.

In [2]:
from pathlib import Path
import ast
import gc
import os
import random
import sys

import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("CUDA disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM total:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2), "GB")

torch: 2.6.0+cu124
CUDA disponivel: True
GPU: NVIDIA GeForce RTX 3060 Ti
VRAM total: 7.66 GB


In [3]:
import importlib.util

required = ["omegaconf", "hydra", "datasets", "huggingface_hub", "transformers"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    print("Dependencias ausentes:", missing)
    print("Reconstruir a imagem com: docker compose up -d --build")
else:
    print("Dependencias essenciais encontradas.")

Dependencias essenciais encontradas.


In [4]:
import subprocess

NCM_COMMIT = "c70274273f489abc73832b2b98584a8483b6b203"
NCM_DIR = Path("/workspace/models/ncm")

if not (NCM_DIR / ".git").exists():
    NCM_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "https://github.com/HazyResearch/ncm.git", str(NCM_DIR)
    ], check=True)

subprocess.run(["git", "-C", str(NCM_DIR), "checkout", NCM_COMMIT], check=True)
sys.path.insert(0, str(NCM_DIR))
print("NCM:", NCM_DIR)
print("commit:", NCM_COMMIT)

NCM: /workspace/models/ncm
commit: c70274273f489abc73832b2b98584a8483b6b203


HEAD is now at c702742 update bibtex


In [5]:
from huggingface_hub import hf_hub_download

CHECKPOINT_DIR = Path("/workspace/models/ncm-checkpoints/xsum")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

for filename in ("config.yaml", "generator.pt", "tokenizer.pt"):
    path = hf_hub_download(
        repo_id="hazyresearch/ncm-checkpoints",
        filename=f"xsum/{filename}",
        local_dir=CHECKPOINT_DIR.parent,
    )
    downloaded = Path(path)
    if downloaded.parent != CHECKPOINT_DIR:
        downloaded.replace(CHECKPOINT_DIR / filename)

print("Checkpoints:", CHECKPOINT_DIR)
print([p.name for p in CHECKPOINT_DIR.iterdir()])

Checkpoints: /workspace/models/ncm-checkpoints/xsum
['tokenizer.pt', 'config.yaml', 'generator.pt']


In [6]:
from omegaconf import OmegaConf

CONFIG_PATH = CHECKPOINT_DIR / "config.yaml"
GENERATOR_PATH = CHECKPOINT_DIR / "generator.pt"
TOKENIZER_PATH = CHECKPOINT_DIR / "tokenizer.pt"

CFG = OmegaConf.load(CONFIG_PATH)
CFG.tokenizer.vqvae.checkpoint_path = str(TOKENIZER_PATH)

from tokenization.vqvae_tokenizer import VQVAETokenizer
from models.next_concept import NextConceptTransformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Este experimento precisa de GPU NVIDIA no container.")

def require_free_vram(min_free_gib):
    torch.cuda.empty_cache()
    free_gib = torch.cuda.mem_get_info()[0] / 2**30
    print("VRAM livre:", round(free_gib, 2), "GiB")
    if free_gib < min_free_gib:
        raise RuntimeError(
            f"VRAM livre insuficiente ({free_gib:.2f} GiB; minimo {min_free_gib:.2f} GiB). "
            "Pare o Ollama/LLM local ou reinicie o kernel antes de executar esta etapa."
        )

require_free_vram(0.8)
tk = VQVAETokenizer.from_config(CFG.tokenizer)
tk.vqvae.to(device)
tk.vqvae.eval()
for param in tk.vqvae.parameters():
    param.requires_grad_(False)

print("VQ-VAE carregado. O gerador sera alocado somente no Stage 2.")
print("VQ-VAE parametros:", round(sum(p.numel() for p in tk.vqvae.parameters()) / 1e6, 2), "M")

VRAM livre: 2.55 GiB
VQVAE, Total parameter count: 60.63M
VQVAE, Non embedding parameter count: 41.30M
VQ-VAE carregado. O gerador sera alocado somente no Stage 2.
VQ-VAE parametros: 60.63 M


## Corpus Python e preservacao estrutural

O corpus combina MBPP com um recorte de CodeSearchNet Python. O texto do codigo nao e normalizado: indentacao, quebras de linha, espacos e pontuacao sao preservados. A saida do checkpoint continua organizada em blocos nativos de 48 tokens.

In [7]:
from datasets import load_dataset

mbpp = load_dataset("google-research-datasets/mbpp", "sanitized")
PAD_ID = int(CFG.model.prefix_pad_token_id)
VQVAE_BLOCK_SIZE = int(tk.vqvae.context_length)
MAX_OUTPUT_BLOCKS = 3
CODESEARCH_TRAIN_ITEMS = 20_000

def normalize_item(prompt, code, source):
    prompt = str(prompt or "").strip()
    code = str(code or "")
    if not prompt or not code or not code.strip():
        return None
    prompt_ids = tk.pre_tokenizer.encode(prompt)
    code_ids = tk.pre_tokenizer.encode(code)
    if not prompt_ids or len(code_ids) < 1:
        return None
    if len(prompt_ids) > CFG.model.prefix_len:
        prompt = tk.pre_tokenizer.decode(prompt_ids[:CFG.model.prefix_len])
    return {"prompt": prompt, "code": code, "source": source}

corpus = [
    normalized
    for split in ("train", "validation", "test")
    for item in mbpp[split]
    if (normalized := normalize_item(item["prompt"], item["code"], "mbpp"))
]

code_search = load_dataset(
    "Nan-Do/code-search-net-python",
    split="train",
    streaming=True,
)
for item in code_search:
    prompt = item.get("summary") or item.get("docstring")
    code = item.get("code") or item.get("original_string")
    normalized = normalize_item(prompt, code, "codesearchnet")
    if normalized:
        corpus.append(normalized)
    if sum(example["source"] == "codesearchnet" for example in corpus) >= CODESEARCH_TRAIN_ITEMS:
        break

random.Random(SEED).shuffle(corpus)
train_end = int(len(corpus) * 0.94)
validation_end = train_end + int(len(corpus) * 0.03)
train_items = corpus[:train_end]
validation_items = corpus[train_end:validation_end]
test_items = corpus[validation_end:]
print("exemplos totais:", len(corpus))
print("treino:", len(train_items))
print("validacao:", len(validation_items))
print("teste:", len(test_items))
print("tamanhos por fonte:")
for source in sorted({item["source"] for item in corpus}):
    print(source, sum(item["source"] == source for item in corpus))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1100 > 1024). Running this sequence through the model will result in indexing errors


exemplos totais: 20420
treino: 19194
validacao: 612
teste: 614
tamanhos por fonte:
codesearchnet 20000
mbpp 420


## Diagnostico do VQ-VAE

Este passo mede o teto do tokenizador congelado antes de qualquer fine-tuning do gerador. `token_accuracy` e `syntax_valid_rate` precisam melhorar apos o Stage 1.

In [8]:
def split_code_blocks(code, max_blocks=MAX_OUTPUT_BLOCKS):
    ids = tk.pre_tokenizer.encode(code)
    return [
        ids[start:start + VQVAE_BLOCK_SIZE]
        for start in range(0, min(len(ids), VQVAE_BLOCK_SIZE * max_blocks), VQVAE_BLOCK_SIZE)
    ]

def padded_code_block(block):
    block = block[:VQVAE_BLOCK_SIZE]
    return block + [tk.pre_tokenizer.pad_token_id] * (VQVAE_BLOCK_SIZE - len(block))

@torch.no_grad()
def evaluate_vqvae(items, max_blocks=MAX_OUTPUT_BLOCKS):
    tk.vqvae.eval()
    exact = correct = total = syntax_valid = indentation_exact = 0
    samples = 0
    for item in items:
        reconstructed_parts = []
        for block in split_code_blocks(item["code"], max_blocks=max_blocks):
            padded = padded_code_block(block)
            source = torch.tensor(padded, device=device).unsqueeze(0)
            reconstruction = tk.vqvae.decode(tk.vqvae.encode(source)).argmax(dim=-1)
            original = source[:, :len(block)]
            decoded = reconstruction[:, :len(block)]
            correct += int((decoded == original).sum())
            total += original.numel()
            exact += int(torch.equal(decoded, original))
            reconstructed_parts.append(reconstruction.squeeze(0).tolist())
        reconstructed = tk.pre_tokenizer.decode(
            [token for part in reconstructed_parts for token in part],
            skip_special_tokens=True,
        ).strip()
        original_code = tk.pre_tokenizer.decode(
            [token for block in split_code_blocks(item["code"], max_blocks) for token in block],
            skip_special_tokens=True,
        ).strip()
        original_indents = [len(line) - len(line.lstrip(" \t")) for line in original_code.splitlines()]
        reconstructed_indents = [len(line) - len(line.lstrip(" \t")) for line in reconstructed.splitlines()]
        indentation_exact += int(original_indents == reconstructed_indents)
        try:
            ast.parse(reconstructed)
            syntax_valid += 1
        except SyntaxError:
            pass
        samples += 1
    return {
        "token_accuracy": correct / max(total, 1),
        "exact_block_rate": exact / max(sum(len(split_code_blocks(item["code"], max_blocks)) for item in items), 1),
        "syntax_valid_rate": syntax_valid / max(samples, 1),
        "indentation_exact_rate": indentation_exact / max(samples, 1),
        "samples": samples,
    }

VQVAE_BASELINE = evaluate_vqvae(test_items[:100])
print("VQ-VAE baseline:", VQVAE_BASELINE)

VQ-VAE baseline: {'token_accuracy': 0.8821767713659605, 'exact_block_rate': 0.017006802721088437, 'syntax_valid_rate': 0.0, 'indentation_exact_rate': 0.13, 'samples': 100}


## Stage 1: adaptar o VQ-VAE ao Python

Treina encoder, decoder e quantizador do VQ-VAE por reconstrução token a token. Ative `RUN_VQVAE_TRAINING=True` somente para esta etapa; o gerador e o Stage 2 permanecem fora dela.

In [9]:
RUN_VQVAE_TRAINING = True
from torch.nn import functional as F
VQVAE_TRAIN_EPOCHS = 8
VQVAE_BATCH_SIZE = 2
VQVAE_ACCUMULATION_STEPS = 8
VQVAE_LEARNING_RATE = 1e-4
VQVAE_LOSS_WEIGHT = 1.0
VQVAE_MAX_TRAIN_ITEMS = 20_000
VQVAE_OUTPUT_DIR = Path("/workspace/results/ncm_codigo_full/vqvae")
VQVAE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vqvae_blocks = [
    padded_code_block(block)
    for item in train_items[:VQVAE_MAX_TRAIN_ITEMS]
    for block in split_code_blocks(item["code"])
]
random.Random(SEED).shuffle(vqvae_blocks)
STRUCTURAL_TOKEN_IDS = set()
for structure in ("\n", "\n    ", "\n\t", "    ", "\t", ":", "(", ")", "[", "]", "{", "}", ","):
    STRUCTURAL_TOKEN_IDS.update(tk.pre_tokenizer.encode(structure))
structural_tokens = torch.tensor(sorted(STRUCTURAL_TOKEN_IDS), device=device)
print("tokens estruturais ponderados:", sorted(STRUCTURAL_TOKEN_IDS))

if RUN_VQVAE_TRAINING:
    require_free_vram(3.5)
    tk.vqvae.train()
    for param in tk.vqvae.parameters():
        param.requires_grad_(True)
    optimizer = torch.optim.AdamW(tk.vqvae.parameters(), lr=VQVAE_LEARNING_RATE, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    history = []

    for epoch in range(1, VQVAE_TRAIN_EPOCHS + 1):
        random.shuffle(vqvae_blocks)
        total_loss = 0.0
        steps = 0
        optimizer.zero_grad(set_to_none=True)
        for start in range(0, len(vqvae_blocks), VQVAE_BATCH_SIZE):
            ids = torch.tensor(
                vqvae_blocks[start:start + VQVAE_BATCH_SIZE],
                dtype=torch.long,
                device=device,
            )
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits, vq_loss, _, utilization, _ = tk.vqvae(ids)
                reconstruction_token_loss = F.cross_entropy(
                    logits.reshape(-1, logits.shape[-1]),
                    ids.reshape(-1),
                    ignore_index=tk.pre_tokenizer.pad_token_id,
                    reduction="none",
                )
                token_weights = torch.isin(ids.reshape(-1), structural_tokens).float() * 3.0 + 1.0
                active_tokens = ids.reshape(-1).ne(tk.pre_tokenizer.pad_token_id).float()
                reconstruction_loss = (
                    reconstruction_token_loss * token_weights * active_tokens
                ).sum() / (token_weights * active_tokens).sum().clamp_min(1.0)
                loss = reconstruction_loss + VQVAE_LOSS_WEIGHT * vq_loss
                loss = loss / VQVAE_ACCUMULATION_STEPS
            scaler.scale(loss).backward()
            steps += 1
            total_loss += loss.item() * VQVAE_ACCUMULATION_STEPS
            if steps % VQVAE_ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(tk.vqvae.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            if steps % 500 == 0:
                print(f"stage1 epoca={epoch} passo={steps} loss={total_loss / steps:.4f} utilization={utilization:.4f}")

        if steps % VQVAE_ACCUMULATION_STEPS != 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        metrics = evaluate_vqvae(test_items[:100])
        history.append({"epoch": epoch, "loss": total_loss / max(steps, 1), **metrics})
        torch.save({
            "model": tk.vqvae.state_dict(),
            "epoch": epoch,
            "history": history,
            "baseline": VQVAE_BASELINE,
        }, VQVAE_OUTPUT_DIR / f"tokenizer_code_epoch_{epoch}.pt")
        print("stage1 metrics:", metrics)

    CODE_VQVAE_CHECKPOINT = VQVAE_OUTPUT_DIR / f"tokenizer_code_epoch_{VQVAE_TRAIN_EPOCHS}.pt"
else:
    candidates = sorted(VQVAE_OUTPUT_DIR.glob("tokenizer_code_epoch_*.pt"))
    CODE_VQVAE_CHECKPOINT = candidates[-1] if candidates else None
    print("Stage 1 desativado. Checkpoint encontrado:", CODE_VQVAE_CHECKPOINT)

if CODE_VQVAE_CHECKPOINT is not None:
    code_vqvae_saved = torch.load(CODE_VQVAE_CHECKPOINT, map_location="cpu", weights_only=False)
    tk.vqvae.load_state_dict(code_vqvae_saved["model"], strict=True)
    print("VQ-VAE de codigo carregado:", CODE_VQVAE_CHECKPOINT)
tk.vqvae.eval()
for param in tk.vqvae.parameters():
    param.requires_grad_(False)
if 'optimizer' in globals():
    del optimizer
if 'scaler' in globals():
    del scaler
gc.collect()
torch.cuda.empty_cache()

tokens estruturais ponderados: [7, 8, 11, 25, 58, 60, 90, 92, 197, 198, 220]
VRAM livre: 6.08 GiB
stage1 epoca=1 passo=500 loss=2.5216 utilization=1.0000
stage1 epoca=1 passo=1000 loss=2.0872 utilization=1.0000
stage1 epoca=1 passo=1500 loss=1.8628 utilization=1.0000
stage1 epoca=1 passo=2000 loss=1.7591 utilization=1.0000
stage1 epoca=1 passo=2500 loss=1.6797 utilization=1.0000
stage1 epoca=1 passo=3000 loss=1.6179 utilization=1.0000
stage1 epoca=1 passo=3500 loss=1.5776 utilization=1.0000
stage1 epoca=1 passo=4000 loss=1.5482 utilization=1.0000
stage1 epoca=1 passo=4500 loss=1.5204 utilization=1.0000
stage1 epoca=1 passo=5000 loss=1.4963 utilization=1.0000
stage1 epoca=1 passo=5500 loss=1.4766 utilization=1.0000
stage1 epoca=1 passo=6000 loss=1.4548 utilization=1.0000
stage1 epoca=1 passo=6500 loss=1.4358 utilization=1.0000
stage1 epoca=1 passo=7000 loss=1.4189 utilization=1.0000
stage1 epoca=1 passo=7500 loss=1.4040 utilization=1.0000
stage1 epoca=1 passo=8000 loss=1.3887 utilizatio

<unknown>:3: SyntaxWarning: invalid decimal literal


stage1 metrics: {'token_accuracy': 0.9869247626004383, 'exact_block_rate': 0.6428571428571429, 'syntax_valid_rate': 0.13, 'indentation_exact_rate': 0.89, 'samples': 100}
stage1 epoca=7 passo=500 loss=0.7844 utilization=1.0000
stage1 epoca=7 passo=1000 loss=0.7896 utilization=1.0000
stage1 epoca=7 passo=1500 loss=0.7869 utilization=1.0000
stage1 epoca=7 passo=2000 loss=0.7873 utilization=1.0000
stage1 epoca=7 passo=2500 loss=0.7876 utilization=1.0000
stage1 epoca=7 passo=3000 loss=0.7871 utilization=1.0000
stage1 epoca=7 passo=3500 loss=0.7860 utilization=1.0000
stage1 epoca=7 passo=4000 loss=0.7856 utilization=1.0000
stage1 epoca=7 passo=4500 loss=0.7841 utilization=1.0000
stage1 epoca=7 passo=5000 loss=0.7828 utilization=1.0000
stage1 epoca=7 passo=5500 loss=0.7826 utilization=1.0000
stage1 epoca=7 passo=6000 loss=0.7813 utilization=1.0000
stage1 epoca=7 passo=6500 loss=0.7798 utilization=1.0000
stage1 epoca=7 passo=7000 loss=0.7790 utilization=1.0000
stage1 epoca=7 passo=7500 loss=0.

In [10]:
require_free_vram(1.5)
net = NextConceptTransformer(vqvae_local=tk.vqvae, config=CFG.model).to(device)
generator_base = torch.load(GENERATOR_PATH, map_location="cpu", weights_only=False)
net.load_state_dict(generator_base["model"], strict=True)
net.eval()

def expand_generator_examples(items):
    expanded = []
    for item in items:
        previous_code = ""
        for block_index, block in enumerate(split_code_blocks(item["code"])):
            prompt_text = item["prompt"]
            if previous_code:
                prompt_text += "\n\n# Continue the Python implementation:\n" + previous_code
            prompt_tokens = tk.pre_tokenizer.encode(prompt_text)
            task_tokens = tk.pre_tokenizer.encode(item["prompt"])
            available_previous = net.prefix_len - min(len(task_tokens), net.prefix_len)
            if len(prompt_tokens) > net.prefix_len and available_previous > 0:
                prompt_tokens = task_tokens[:net.prefix_len - available_previous] + prompt_tokens[-available_previous:]
            expanded.append({
                **item,
                "prompt_ids": prompt_tokens[:net.prefix_len],
                "code": tk.pre_tokenizer.decode(block, skip_special_tokens=True),
                "block_index": block_index,
            })
            previous_code += tk.pre_tokenizer.decode(block, skip_special_tokens=True)
    return expanded

generator_train_items = expand_generator_examples(train_items)
generator_validation_items = expand_generator_examples(validation_items)
generator_test_items = expand_generator_examples(test_items)
print("pares gerador:", len(generator_train_items), len(generator_validation_items), len(generator_test_items))

@torch.no_grad()
def make_training_batch(items):
    prefix_chunks = []
    ncm_chunks = []
    target_chunks = []
    prefix_mask_chunks = []

    for item in items:
        prompt_ids = list(item.get("prompt_ids", tk.pre_tokenizer.encode(item["prompt"])))
        prompt_ids = prompt_ids[:net.prefix_len]
        prompt_ids = prompt_ids + [PAD_ID] * (net.prefix_len - len(prompt_ids))
        prefix = torch.tensor(prompt_ids, dtype=torch.long, device=device).unsqueeze(0)
        prefix_mask_chunks.append(prefix.eq(PAD_ID))

        code_ids = tk.pre_tokenizer.encode(item["code"])
        code_ids = padded_code_block(code_ids)
        code_tensor = torch.tensor(code_ids, dtype=torch.long, device=device).unsqueeze(0)

        target = tk.vqvae.encode(code_tensor)
        ncm_input = tk.vqvae.vq.idx_to_ncm_input(target)
        if net.query_level0:
            l0_query = net.build_l0_query_input(1, dtype=ncm_input.dtype)
            ncm_input = torch.cat([l0_query, ncm_input], dim=1)
        prefix_embedding = tk.vqvae.encoder.wte(prefix)
        ncm_chunks.append(ncm_input)
        prefix_chunks.append(prefix_embedding)
        target_chunks.append(target)

    prefix_embedding = torch.cat(prefix_chunks, dim=0)
    ncm_input = torch.cat(ncm_chunks, dim=0)
    model_input = torch.cat([prefix_embedding, ncm_input], dim=1)
    targets = torch.cat(target_chunks, dim=0)
    prefix_pad_mask = torch.cat(prefix_mask_chunks, dim=0)
    return model_input, targets, prefix_pad_mask

smoke_items = generator_train_items[:1]
smoke_input, smoke_targets, smoke_prefix_pad_mask = make_training_batch(smoke_items)
net.eval()
with torch.inference_mode():
    smoke_logits = net.forward(smoke_input, prefix_pad_mask=smoke_prefix_pad_mask)

print("entrada:", tuple(smoke_input.shape))
print("alvos:", tuple(smoke_targets.shape))
print("logits:", tuple(smoke_logits.shape))
assert smoke_input.shape[1] == net.prefix_len + sum(net.levels)
assert smoke_logits.shape[:2] == smoke_targets.shape
print("Smoke test: OK")

VRAM livre: 5.84 GiB
NCP Transformer, Total parameter count: 88.02M
NCP Transformer, Non embedding parameter count: 87.71M
pares gerador: 55947 1779 1779
entrada: (1, 712, 384)
alvos: (1, 200)
logits: (1, 200, 4096)
Smoke test: OK


## Stage 2: fine-tuning do gerador NCM

A celula abaixo treina somente o gerador sobre o VQ-VAE já adaptado. Cada exemplo inclui o bloco de codigo alvo de 48 tokens e os blocos anteriores no prefixo. Checkpoints sao gravados em `/workspace/results/ncm_codigo_full/generator`.

In [ ]:
import matplotlib.pyplot as plt
from torch.nn import functional as F

RUN_GENERATOR_TRAINING = True
BATCH_SIZE = 1
ACCUMULATION_STEPS = 8
LEARNING_RATE = 5e-5
TOTAL_EPOCHS = 20
MAX_TRAIN_ITEMS = len(generator_train_items)
PATIENCE = 12

OUTPUT_DIR = Path("/workspace/results/ncm_codigo_full/generator")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_loss_history = []
validation_loss_history = []

if RUN_GENERATOR_TRAINING:
    if not torch.cuda.is_available():
        raise RuntimeError("Treino interrompido: CUDA indisponivel.")

    net.train()
    net.vqvae_local[0].eval()
    optimizer = torch.optim.AdamW(
        (p for p in net.parameters() if p.requires_grad),
        lr=LEARNING_RATE,
        weight_decay=0.01,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    optimizer.zero_grad(set_to_none=True)

    checkpoints = list(OUTPUT_DIR.glob("epoch_*.pt"))
    if checkpoints:
        latest_path = max(checkpoints, key=lambda path: int(path.stem.split("_")[-1]))
        saved = torch.load(latest_path, map_location="cpu", weights_only=False)
        net.load_state_dict(saved["model"], strict=True)
        start_epoch = int(saved.get("epoch", 0))
        train_loss_history = list(saved.get("train_loss_history", []))
        validation_loss_history = list(saved.get("validation_loss_history", []))
        if saved.get("optimizer") is not None:
            optimizer.load_state_dict(saved["optimizer"])
        print("retomando de:", latest_path, "epoca:", start_epoch)
    else:
        start_epoch = 0

    selected = generator_train_items[:MAX_TRAIN_ITEMS]
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=2)
    best_val_loss = min(validation_loss_history) if validation_loss_history else float("inf")
    bad_epochs = 0

    for epoch in range(start_epoch + 1, TOTAL_EPOCHS + 1):
        random.shuffle(selected)
        epoch_loss = 0.0
        steps = 0

        for start in range(0, len(selected), BATCH_SIZE):
            batch_items = selected[start:start + BATCH_SIZE]
            model_input, targets, prefix_pad_mask = make_training_batch(batch_items)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = net.forward(model_input, prefix_pad_mask=prefix_pad_mask)
                loss = F.cross_entropy(
                    logits.float().reshape(-1, logits.shape[-1]),
                    targets.reshape(-1),
                ) / ACCUMULATION_STEPS

            scaler.scale(loss).backward()
            steps += 1

            if steps % ACCUMULATION_STEPS == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            epoch_loss += loss.item() * ACCUMULATION_STEPS
            if steps % 8 == 0:
                print(f"epoca={epoch} passo={steps} loss={epoch_loss / steps:.4f}")

        if steps % ACCUMULATION_STEPS != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        mean_loss = epoch_loss / max(steps, 1)
        net.eval()
        validation_loss = 0.0
        validation_steps = 0
        with torch.inference_mode():
            for start in range(0, len(generator_validation_items), BATCH_SIZE):
                batch_items = generator_validation_items[start:start + BATCH_SIZE]
                model_input, targets, prefix_pad_mask = make_training_batch(batch_items)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = net.forward(model_input, prefix_pad_mask=prefix_pad_mask)
                    loss = F.cross_entropy(
                        logits.float().reshape(-1, logits.shape[-1]),
                        targets.reshape(-1),
                    )
                validation_loss += loss.item()
                validation_steps += 1
        mean_validation_loss = validation_loss / max(validation_steps, 1)
        scheduler.step(mean_validation_loss)
        train_loss_history.append(mean_loss)
        validation_loss_history.append(mean_validation_loss)
        net.train()
        net.vqvae_local[0].eval()

        checkpoint_path = OUTPUT_DIR / f"epoch_{epoch}.pt"
        torch.save({
            "model": net.state_dict(),
            "epoch": epoch,
            "loss": mean_loss,
            "validation_loss": mean_validation_loss,
            "train_loss_history": train_loss_history,
            "validation_loss_history": validation_loss_history,
            "optimizer": optimizer.state_dict(),
            "config": OmegaConf.to_container(CFG, resolve=True),
        }, checkpoint_path)
        print(
            f"epoca={epoch} train_loss={mean_loss:.4f} "
            f"validation_loss={mean_validation_loss:.4f} checkpoint={checkpoint_path}"
        )

        if mean_validation_loss < best_val_loss:
            best_val_loss = mean_validation_loss
            bad_epochs = 0
            torch.save({
                "model": net.state_dict(),
                "epoch": epoch,
                "validation_loss": mean_validation_loss,
                "config": OmegaConf.to_container(CFG, resolve=True),
            }, OUTPUT_DIR / "best.pt")
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"Parada antecipada sem melhoria por {PATIENCE} epocas.")
                break

    plt.figure(figsize=(8, 4))
    plt.plot(range(1, len(train_loss_history) + 1), train_loss_history, marker="o", label="treino")
    plt.plot(range(1, len(validation_loss_history) + 1), validation_loss_history, marker="o", label="validacao")
    plt.xlabel("Epoca")
    plt.ylabel("Cross entropy")
    plt.legend()
    plt.title("Perda de treino NCM - Python")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "loss_curve.png")
    plt.show()
else:
    print("Stage 2 desativado. Ajuste RUN_GENERATOR_TRAINING=True depois do Stage 1 e do smoke test.")

epoca=1 passo=8 loss=22.0164
epoca=1 passo=16 loss=20.0933
epoca=1 passo=24 loss=19.5915
epoca=1 passo=32 loss=18.0132
epoca=1 passo=40 loss=16.3414
epoca=1 passo=48 loss=15.4495
epoca=1 passo=56 loss=14.6015
epoca=1 passo=64 loss=13.8973
epoca=1 passo=72 loss=13.2581
epoca=1 passo=80 loss=12.8232
epoca=1 passo=88 loss=12.4110
epoca=1 passo=96 loss=12.0248
epoca=1 passo=104 loss=11.7088
epoca=1 passo=112 loss=11.5111
epoca=1 passo=120 loss=11.3663
epoca=1 passo=128 loss=11.1747
epoca=1 passo=136 loss=10.9898
epoca=1 passo=144 loss=10.8034
epoca=1 passo=152 loss=10.6298
epoca=1 passo=160 loss=10.5009
epoca=1 passo=168 loss=10.3728
epoca=1 passo=176 loss=10.2609
epoca=1 passo=184 loss=10.1038
epoca=1 passo=192 loss=10.0022
epoca=1 passo=200 loss=9.8965
epoca=1 passo=208 loss=9.7790
epoca=1 passo=216 loss=9.6781
epoca=1 passo=224 loss=9.5914
epoca=1 passo=232 loss=9.5461
epoca=1 passo=240 loss=9.4859
epoca=1 passo=248 loss=9.4148
epoca=1 passo=256 loss=9.3392
epoca=1 passo=264 loss=9.2802

## Geracao e validacao sintetica

A geração carrega explicitamente `best.pt`, escolhido pela menor perda de validacao. O resultado pode ser sintaticamente inválido porque o VQ-VAE XSum não foi treinado em código.

In [10]:
NEW_BEST_CHECKPOINT = Path("/workspace/results/ncm_codigo_full/generator/best.pt")
LEGACY_BEST_CHECKPOINT = Path("/workspace/results/ncm_codigo/best.pt")
if CODE_VQVAE_CHECKPOINT is not None and not NEW_BEST_CHECKPOINT.exists():
    raise FileNotFoundError("O Stage 1 foi carregado, execute o Stage 2 antes da inferencia.")
BEST_CHECKPOINT = NEW_BEST_CHECKPOINT if NEW_BEST_CHECKPOINT.exists() else LEGACY_BEST_CHECKPOINT
if not BEST_CHECKPOINT.exists():
    raise FileNotFoundError(f"Checkpoint fine-tunado nao encontrado: {BEST_CHECKPOINT}")
best_saved = torch.load(BEST_CHECKPOINT, map_location="cpu", weights_only=False)
net.load_state_dict(best_saved["model"], strict=True)
print(
    "checkpoint de inferencia:", BEST_CHECKPOINT,
    "epoca:", best_saved.get("epoch"),
    "validation_loss:", round(best_saved.get("validation_loss", 0.0), 4),
)

if test_items:
    sample = test_items[0]
else:
    sample = train_items[0]

prefix_ids = tk.pre_tokenizer.encode(sample["prompt"])
prefix_ids = prefix_ids[:512]
prefix_ids = prefix_ids + [PAD_ID] * (512 - len(prefix_ids))
prefix_tensor = torch.tensor(prefix_ids, dtype=torch.long, device=device).unsqueeze(0)

net.eval()
with torch.inference_mode():
    concept_ids = net.generate(
        idx=None,
        prefix_tokens=prefix_tensor,
        temperature=0.8,
        top_k=40,
        top_p=0.9,
    )
    logits = tk.vqvae.decode(concept_ids)
    raw_ids = logits.squeeze(0).argmax(dim=-1)
    generated = tk.pre_tokenizer.decode(raw_ids.tolist(), skip_special_tokens=True)

print("prompt:", sample["prompt"])
print("codigo gerado:\n", generated)
try:
    ast.parse(generated)
    print("Sintaxe Python: valida")
except SyntaxError as exc:
    print("Sintaxe Python: invalida -", exc)

checkpoint de inferencia: /workspace/results/ncm_codigo/best.pt epoca: 57 validation_loss: 3.2365
prompt: Write a function to check whether an element exists within a tuple.
codigo gerado:
 def check_11(1,):):
 resorted north(1(sum for in in_up11++1) return 1. return.cern
cedcern playbackistar

Sintaxe Python: invalida - unmatched ')' (<unknown>, line 1)


## Inferencia interativa

Edite `INFERENCE_PROMPT` e execute a celula para ver os codigos NCM gerados em cada nivel, os tokens GPT-2 obtidos pelo decodificador VQ-VAE e o codigo Python final.

In [11]:
from IPython.display import Code, Markdown, display

INFERENCE_PROMPT = "Write a Python function that returns the square of an integer."
INFERENCE_TEMPERATURE = 0.3
INFERENCE_TOP_K = 20
INFERENCE_TOP_P = 0.9
INFERENCE_BLOCKS = 3
INFERENCE_CANDIDATES = 5

def generate_and_show(prompt, temperature=0.3, top_k=20, top_p=0.9, blocks=3):
    net.eval()
    generated_parts = []
    all_concepts = []
    all_token_ids = []
    for block_index in range(blocks):
        generated_so_far = "".join(generated_parts)
        condition = prompt
        if generated_so_far:
            condition += "\n\n# Continue the Python implementation:\n" + generated_so_far
        prompt_ids = tk.pre_tokenizer.encode(condition)
        task_ids = tk.pre_tokenizer.encode(prompt)
        if len(prompt_ids) > net.prefix_len:
            available = max(net.prefix_len - min(len(task_ids), net.prefix_len), 0)
            prompt_ids = task_ids[:net.prefix_len - available] + prompt_ids[-available:] if available else task_ids[:net.prefix_len]
        prompt_ids += [PAD_ID] * (net.prefix_len - len(prompt_ids))
        prefix_tensor = torch.tensor(
            prompt_ids, dtype=torch.long, device=device
        ).unsqueeze(0)
        with torch.inference_mode():
            concept_ids = net.generate(
                idx=None,
                prefix_tokens=prefix_tensor,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
            )
            token_logits = tk.vqvae.decode(concept_ids)
            raw_ids = token_logits.squeeze(0).argmax(dim=-1)
        generated_part = tk.pre_tokenizer.decode(
            raw_ids.tolist(), skip_special_tokens=True
        )
        generated_parts.append(generated_part)
        all_concepts.append(concept_ids.detach().cpu())
        all_token_ids.append(raw_ids.detach().cpu())
    generated = "".join(generated_parts)

    display(Markdown("### Prompt"), Markdown(f"{prompt}"))
    display(Markdown("### Codigos NCM gerados por nivel"))
    for block_index, concept_ids in enumerate(all_concepts):
        offset = 0
        print(f"bloco {block_index + 1}")
        for level, width in enumerate(net.levels):
            values = concept_ids[0, offset:offset + width].tolist()
            print(f"  nivel {level}: {values}")
            offset += width

    display(Markdown("### Tokens GPT-2 decodificados"))
    for block_index, token_ids in enumerate(all_token_ids):
        print(f"bloco {block_index + 1}: {token_ids.tolist()}")
    display(Markdown("### Codigo gerado"))
    display(Code(generated, language="python"))

    try:
        ast.parse(generated)
        display(Markdown("**Sintaxe Python: valida**"))
    except SyntaxError as exc:
        display(Markdown(f"**Sintaxe Python: invalida** — `{exc}`"))
    return generated, all_concepts, all_token_ids

valid_candidate = None
inference_candidates = []
for candidate_index in range(INFERENCE_CANDIDATES):
    display(Markdown(f"## Candidato {candidate_index + 1}"))
    candidate = generate_and_show(
        INFERENCE_PROMPT,
        temperature=INFERENCE_TEMPERATURE,
        top_k=INFERENCE_TOP_K,
        top_p=INFERENCE_TOP_P,
        blocks=INFERENCE_BLOCKS,
    )
    inference_candidates.append(candidate)
    try:
        parsed_ast = ast.parse(candidate[0])
        valid_candidate = candidate
        print("AST aceito:", type(parsed_ast.body[0]).__name__ if parsed_ast.body else "vazio")
        break
    except SyntaxError:
        continue

if valid_candidate is None:
    print("Nenhum candidato produziu uma AST valida.")

### Prompt

Write a Python function that returns the square of an integer.

### Codigos NCM gerados por nivel

nivel 0: [383, 2802, 1590, 3396]
nivel 1: [2459, 3514, 2463, 2463, 2525, 2525]
nivel 2: [1781, 175, 3456, 1781, 1509, 311, 1509, 2668, 2668, 2668]
nivel 3: [679, 1095, 693, 693, 1320, 3285, 2037, 839, 1095, 3129, 3129, 3129, 3129, 3129]
nivel 4: [3914, 2158, 2662, 1841, 3760, 1302, 1651, 3226, 1841, 1841, 1841, 1841, 1302, 258, 3740, 3740, 3740, 3740, 3740, 3740]
nivel 5: [5, 965, 1093, 1093, 1811, 310, 2771, 2465, 2465, 1093, 2416, 2416, 2416, 2634, 2634, 2416, 3406, 2572, 2572, 85, 85, 3406, 85, 3406, 85, 85]
nivel 6: [3292, 502, 1947, 3670, 1815, 601, 495, 2943, 2943, 1815, 2997, 20, 568, 919, 3771, 3076, 1815, 568, 601, 1815, 4022, 568, 2617, 2617, 568, 568, 568, 568, 568, 568, 568, 2617]
nivel 7: [3244, 1657, 1310, 1181, 3868, 411, 411, 3868, 411, 1767, 304, 2094, 411, 304, 1657, 3036, 1550, 1548, 2971, 3868, 343, 3868, 3868, 1000, 3868, 3888, 3137, 3137, 3137, 3793, 3137, 3137, 3793, 3888, 3793, 3888, 3888, 3793, 3793, 3137]
nivel 8: [3985, 2324, 2159, 1425, 1105, 1952, 748, 457,

### Tokens GPT-2 decodificados

[4299, 6616, 62, 1870, 18301, 7, 16, 77, 2599, 198, 220, 220, 220, 220, 1441, 7530, 1581, 7, 10641, 1343, 25, 1343, 1343, 14, 77, 9, 9, 77, 1343, 8, 50256, 50256, 352, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256]


### Codigo gerado

def square_And Eyes(1n):
     return scheduledOR(char +: + +/n**n +) 1

**Sintaxe Python: invalida** — `invalid decimal literal (<unknown>, line 1)`

## Limitacoes

- o checkpoint XSum e apenas a inicializacao; avalie se o Stage 1 melhorou a reconstrucao antes do Stage 2;
- o VQ-VAE gera blocos de 48 tokens por chamada; sequencias maiores dependem de continuacao condicional entre blocos;
- o modelo pode produzir uma AST valida que ainda contenha erros semanticos ou falhas nos testes;
- candidates validos precisam ser executados em testes antes de uso;
- nao use o checkpoint como modelo de producao sem validacao humana e testes executaveis.